# GPU Preprocessing Runner (Local VS Code Notebook)

This notebook runs the project CLI scripts on a local machine with GPU support.

What you need to edit:
- `REPO_DIR` (local repository path)
- `INPUT_DIR` (folder with your 2D slices)

Pipeline executed by the main script:
1. Stack slices into a 3D volume
2. Apply norm200 normalization
3. Run CUDA NLM (chunked)
4. Save outputs into `norm200_output/` and `nlm_output/`

## 1) Local runtime setup
Use the VS Code Jupyter kernel from `venv-napari` for local GPU execution.

In [13]:
import os
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version)
print('Python executable:', sys.executable)
print('Working dir:', os.getcwd())

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
Python executable: c:\Users\rony.schwartz\.conda\envs\venv-napari\python.exe
Working dir: c:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess


In [14]:
# Local GPU check (PyTorch-based)
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    print('GPU name:', torch.cuda.get_device_name(0))

CUDA available: True
GPU count: 2
GPU name: NVIDIA RTX A6000


In [15]:
# Optional local dependency check (no install command needed)
required = ['numpy', 'tifffile', 'torch']
for pkg in required:
    try:
        __import__(pkg)
        print(f'OK: {pkg}')
    except Exception as e:
        print(f'MISSING: {pkg} -> {e}')

OK: numpy
OK: tifffile
OK: torch


## 2) Confirm paths
Defaults below are set for your local workspace, input slices folder, and export target folder.

In [16]:
REPO_DIR = Path(r'C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT')
INPUT_DIR = Path(r'\\HIVE3065\Yael_Mishael\Rony\10.12.25_Rehovot_samp_2\Rehovot_samp2_highkV_Cu0.11_15um_Rec')
EXPORT_DIR = Path(r'\\HIVE3065\Yael_Mishael\Rony\remote_computer backup\10.5')
EXPORT_BASENAME = 'rehovot_samp_2'

print('REPO_DIR =', REPO_DIR)
print('INPUT_DIR =', INPUT_DIR)
print('EXPORT_DIR =', EXPORT_DIR)
print('EXPORT_BASENAME =', EXPORT_BASENAME)

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Repository not found: {REPO_DIR}')
if not INPUT_DIR.exists():
    raise FileNotFoundError(f'Input folder not found: {INPUT_DIR}')
if not EXPORT_DIR.exists():
    raise FileNotFoundError(f'Export folder not found: {EXPORT_DIR}')

patterns = ('*.tif', '*.tiff', '*.png')
direct_slices = []
for p in patterns:
    direct_slices.extend(sorted(INPUT_DIR.glob(p)))
print('Direct slice count in INPUT_DIR:', len(direct_slices))

REPO_DIR = C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT
INPUT_DIR = \\HIVE3065\Yael_Mishael\Rony\10.12.25_Rehovot_samp_2\Rehovot_samp2_highkV_Cu0.11_15um_Rec
EXPORT_DIR = \\HIVE3065\Yael_Mishael\Rony\remote_computer backup\10.5
EXPORT_BASENAME = rehovot_samp_2
Direct slice count in INPUT_DIR: 812


## 3) Quick CLI check
This verifies the script is callable and shows CLI help using the current kernel interpreter.

In [17]:
# If slices are inside subfolders, auto-select the first folder that contains images.
slice_patterns = ('*.tif', '*.tiff', '*.png')

def count_slices(folder: Path) -> int:
    c = 0
    for pattern in slice_patterns:
        c += len(list(folder.glob(pattern)))
    return c

if count_slices(INPUT_DIR) == 0:
    candidate_folders = []
    for subdir in sorted([p for p in INPUT_DIR.rglob('*') if p.is_dir()]):
        n = count_slices(subdir)
        if n > 0:
            candidate_folders.append((subdir, n))

    if not candidate_folders:
        raise FileNotFoundError(
            f'No .tif/.tiff/.png slices found under INPUT_DIR (recursive): {INPUT_DIR}'
        )

    INPUT_DIR = candidate_folders[0][0]
    print('No direct slices found. Using nested folder:', INPUT_DIR)
    print('Slice count in selected folder:', candidate_folders[0][1])
else:
    print('Using INPUT_DIR as-is for slice loading.')

Using INPUT_DIR as-is for slice loading.


In [18]:
# Build a uniform-shape slice set when mixed image sizes are present.
import shutil
from collections import Counter
import imageio.v3 as iio
import tifffile

slice_files = []
for pattern in ('*.tif', '*.tiff', '*.png'):
    slice_files.extend(sorted(INPUT_DIR.glob(pattern)))

if not slice_files:
    raise FileNotFoundError(f'No slices found in {INPUT_DIR}')

shape_by_file = {}
read_errors = []

for p in slice_files:
    try:
        if p.suffix.lower() in ('.tif', '.tiff'):
            arr = tifffile.imread(str(p))
        else:
            arr = iio.imread(str(p))
        shape_by_file[p] = tuple(arr.shape)
    except Exception as e:
        read_errors.append((p, str(e)))

if read_errors:
    print(f'Read errors: {len(read_errors)} files (showing up to 5)')
    for p, e in read_errors[:5]:
        print(' -', p, '|', e)

shape_counter = Counter(shape_by_file.values())
if not shape_counter:
    raise RuntimeError('No readable slices were found.')

dominant_shape, dominant_count = shape_counter.most_common(1)[0]
print('Shape distribution:', dict(shape_counter))
print('Dominant shape:', dominant_shape, 'count:', dominant_count, 'out of', len(slice_files))

mismatched = [p for p, s in shape_by_file.items() if s != dominant_shape]
print('Mismatched slice count:', len(mismatched))

if mismatched:
    CLEAN_INPUT_DIR = REPO_DIR / 'preprocess' / '_tmp_uniform_slices' / EXPORT_BASENAME
    if CLEAN_INPUT_DIR.exists():
        shutil.rmtree(CLEAN_INPUT_DIR)
    CLEAN_INPUT_DIR.mkdir(parents=True, exist_ok=True)

    for p, s in shape_by_file.items():
        if s == dominant_shape:
            shutil.copy2(p, CLEAN_INPUT_DIR / p.name)

    EFFECTIVE_INPUT_DIR = CLEAN_INPUT_DIR
    print('Created cleaned input folder:', EFFECTIVE_INPUT_DIR)
    print('Kept slices:', dominant_count)
    print('Dropped mismatched slices (showing up to 10):')
    for p in mismatched[:10]:
        print(' -', p.name, shape_by_file[p])
else:
    EFFECTIVE_INPUT_DIR = INPUT_DIR
    print('All slices have uniform shape. Using INPUT_DIR directly.')

Shape distribution: {(1236, 1236): 811, (896, 1236): 1}
Dominant shape: (1236, 1236) count: 811 out of 812
Mismatched slice count: 1
Created cleaned input folder: C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess\_tmp_uniform_slices\rehovot_samp_2
Kept slices: 811
Dropped mismatched slices (showing up to 10):
 - Rehovot_samp2_highkV_Cu0.11_15um_rec_spr.tif (896, 1236)


In [19]:
cmd = [sys.executable, 'preprocess/run_preprocess.py', '--help']
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
print('Return code:', result.returncode)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('CLI help command failed.')

Running: c:\Users\rony.schwartz\.conda\envs\venv-napari\python.exe preprocess/run_preprocess.py --help
Return code: 0
usage: run_preprocess.py [-h] --input_dir INPUT_DIR

Run stack -> norm200 -> CUDA NLM and save 3D TIFF outputs.

options:
  -h, --help            show this help message and exit
  --input_dir INPUT_DIR
                        Path to folder containing 2-D slice images
                        (.tif/.tiff/.png).



## 4) Run the main GPU preprocessing CLI
This executes stack -> norm200 -> CUDA NLM (chunked).

In [20]:
# Optional: center-crop to a 650x650x650 volume and write it as slices
CROP_SIZE = 650

base_input_dir = EFFECTIVE_INPUT_DIR if 'EFFECTIVE_INPUT_DIR' in globals() else INPUT_DIR
crop_source_files = []
for pattern in ('*.tif', '*.tiff', '*.png'):
    crop_source_files.extend(sorted(base_input_dir.glob(pattern)))

if len(crop_source_files) < CROP_SIZE:
    raise ValueError(
        f'Not enough slices for Z crop: found {len(crop_source_files)}, need at least {CROP_SIZE}'
    )

z_center = len(crop_source_files) // 2
z_start = z_center - (CROP_SIZE // 2)
z_end = z_start + CROP_SIZE
selected_files = crop_source_files[z_start:z_end]

# Use first slice to get XY shape and build centered XY crop indices.
first_img = tifffile.imread(str(selected_files[0]))
if first_img.ndim != 2:
    raise ValueError(f'Expected 2D slices, got shape {first_img.shape}')

h, w = first_img.shape
if h < CROP_SIZE or w < CROP_SIZE:
    raise ValueError(
        f'Slice size too small for XY crop: got {(h, w)}, need at least ({CROP_SIZE}, {CROP_SIZE})'
    )

y_start = (h - CROP_SIZE) // 2
y_end = y_start + CROP_SIZE
x_start = (w - CROP_SIZE) // 2
x_end = x_start + CROP_SIZE

CROPPED_INPUT_DIR = REPO_DIR / 'preprocess' / '_tmp_center_crop_650' / EXPORT_BASENAME
if CROPPED_INPUT_DIR.exists():
    shutil.rmtree(CROPPED_INPUT_DIR)
CROPPED_INPUT_DIR.mkdir(parents=True, exist_ok=True)

for i, src in enumerate(selected_files):
    img = tifffile.imread(str(src))
    if img.shape != (h, w):
        raise ValueError(f'Unexpected shape change in {src.name}: {img.shape} vs {(h, w)}')
    cropped = img[y_start:y_end, x_start:x_end]
    out_name = f'slice_{i:06d}.tif'
    tifffile.imwrite(str(CROPPED_INPUT_DIR / out_name), cropped)

print('Created cropped input folder:', CROPPED_INPUT_DIR)
print('Cropped volume size (Z, Y, X):', (CROP_SIZE, CROP_SIZE, CROP_SIZE))
print('Source folder used:', base_input_dir)

Created cropped input folder: C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess\_tmp_center_crop_650\rehovot_samp_2
Cropped volume size (Z, Y, X): (650, 650, 650)
Source folder used: C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess\_tmp_uniform_slices\rehovot_samp_2


In [21]:
if 'CROPPED_INPUT_DIR' in globals():
    RUN_INPUT_DIR = CROPPED_INPUT_DIR
elif 'EFFECTIVE_INPUT_DIR' in globals():
    RUN_INPUT_DIR = EFFECTIVE_INPUT_DIR
else:
    RUN_INPUT_DIR = INPUT_DIR

cmd = [
    sys.executable,
    'preprocess/run_preprocess.py',
    '--input_dir',
    str(RUN_INPUT_DIR),
]

print('Running:', ' '.join(cmd))
print('Using input folder:', RUN_INPUT_DIR)
result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
print('Return code:', result.returncode)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Preprocess CLI failed. See printed stdout/stderr above.')

Running: c:\Users\rony.schwartz\.conda\envs\venv-napari\python.exe preprocess/run_preprocess.py --input_dir C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess\_tmp_center_crop_650\rehovot_samp_2
Using input folder: C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess\_tmp_center_crop_650\rehovot_samp_2
Return code: 0
CUDA device: NVIDIA RTX A6000
Input dir: C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess\_tmp_center_crop_650\rehovot_samp_2
Step 1/3: stacking slices
Stacked 650 slices -> volume (650, 650, 650), dtype=uint16
Step 2/3: applying norm200
norm200: detected mode = 188, rescaled to target = 200.0
Saving norm200 output
Step 3/3: applying CUDA NLM (chunked)
NLM CUDA: estimated sigma=0.232565, h=0.139539
NLM CUDA: tile 1/216 core=[0:128, 0:128, 0:128]
NLM CUDA: tile 2/216 core=[0:128, 0:128, 128:256]
NLM CUDA: tile 3/216 core=[0:128, 0:128, 256:384]
NLM CUDA: tile 4/216 core=[0:128, 0:128, 384:512]
NLM CUDA: tile 5/216 core=[0:128, 0:128, 512:640

## 5) Validate outputs and export final TIF
Expected intermediate outputs:
- `preprocess/norm200_output/norm200_volume.tif`
- `preprocess/nlm_output/nlm_volume.tif`

Final export target:
- `\\HIVE3065\Yael_Mishael\Rony\remote_computer backup\10.5\rehovot_samp_2.tif`

In [22]:
import shutil
import tifffile

norm_path = REPO_DIR / 'preprocess' / 'norm200_output' / 'norm200_volume.tif'
nlm_path = REPO_DIR / 'preprocess' / 'nlm_output' / 'nlm_volume.tif'
export_path = EXPORT_DIR / f'{EXPORT_BASENAME}.tif'

print('norm path exists:', norm_path.exists(), norm_path)
print('nlm path exists:', nlm_path.exists(), nlm_path)
print('export path:', export_path)

if norm_path.exists():
    norm_vol = tifffile.imread(norm_path)
    print('norm200 shape:', norm_vol.shape, 'dtype:', norm_vol.dtype)

if nlm_path.exists():
    nlm_vol = tifffile.imread(nlm_path)
    print('nlm shape:', nlm_vol.shape, 'dtype:', nlm_vol.dtype)

    shutil.copy2(nlm_path, export_path)
    print('Exported final TIF to:', export_path)
else:
    raise FileNotFoundError(f'Expected NLM output not found: {nlm_path}')

norm path exists: True C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess\norm200_output\norm200_volume.tif
nlm path exists: True C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT\preprocess\nlm_output\nlm_volume.tif
export path: \\HIVE3065\Yael_Mishael\Rony\remote_computer backup\10.5\rehovot_samp_2.tif
norm200 shape: (650, 650, 650) dtype: float32
nlm shape: (650, 650, 650) dtype: float32
Exported final TIF to: \\HIVE3065\Yael_Mishael\Rony\remote_computer backup\10.5\rehovot_samp_2.tif


## Optional: run the playground CLI (desktop workflow)
This command is optional and can be used on a local machine with a GUI setup.

In [23]:
# Example only (optional):
# subprocess.run([
#     sys.executable, 'preprocess_playground/run_napari_filters.py',
#     '--input_dir', str(INPUT_DIR),
#     '--filter', 'nlm'
# ], cwd=str(REPO_DIR), check=True)